In [0]:
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.appName("ShoppingBadshah").getOrCreate()

In [0]:
customers = [
    (1, "alice@example.com", "Alice", "US", "2019-06-01"),
    (2, "bob@example.com", "Bob", "UK", "2020-01-15"),
    (3, "carla@example.com", "Carla", "IN", "2021-11-09"),
    (4, "dan@example.com", "Dan", "US", "2022-08-20"),
]

customers_schema = ["customer_id", "email", "name", "country", "signup_date"]
customer_dataframe = spark.createDataFrame(data=customers, schema=customers_schema)

products = [
    (101, "Echo Dot", "electronics", 49.99),
    (102, "Kindle", "electronics", 129.99),
    (103, "Coffee Mug", "home", 9.99),
    (104, "Yoga Mat", "sports", 19.99)
]

product_schema = ["product_id", "title", "category", "price"]
products_dataframe = spark.createDataFrame(products, product_schema)

orders = [
    (1001, 1, 101, 2, "2023-07-01", "shipped"),
    (1002, 2, 103, 1, "2023-07-02", "delivered"),
    (1003, 1, 102, 1, "2023-07-15", "cancelled"),
    (1004, 3, 104, 3, "2023-07-20", "delivered"),
]
orders_schema = ["order_id", "customer_id", "product_id", "quantity", "order_date", "status"]
orders_df = spark.createDataFrame(orders, orders_schema)



In [0]:
# Explore the data

orders_df.printSchema()
orders_df.show()

# total order count
print("Total orders:", orders_df.count())

In [0]:
customer_dataframe.show()

In [0]:
# Select + Filter + add colums
from pyspark.sql import functions as F
customer_dataframe.withColumn("signup_year", F.year(F.to_date("signup_date"))) \
    .select("name", "country", "signup_year").filter(F.col("country") == "US") \
    .show()

In [0]:
# Aggregation (Total Revenue per Product)

orders_enriched = orders_df.join(products_dataframe, "product_id", "left")
orders_enriched.show()


revenue_by_product = orders_enriched.withColumn(
    "revenue", F.col("quantity") * F.col("price")
).groupBy("product_id", "title").agg(F.sum("revenue").alias("total_revenue"))

revenue_by_product.show()



In [0]:
# Joins (with Broadcast Hint)
from pyspark.sql.functions import broadcast

orders_with_customer = orders_df.join(broadcast(customer_dataframe), orders_df.customer_id == customer_dataframe.customer_id, "left")

orders_with_customer.select("order_id", "name", "country", "status").show()

In [0]:
# Window Functions (Ranking)
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy(F.to_date("order_date").desc())
orders_df.withColumn("rank", F.row_number().over(w)).show()

In [0]:
# UDF vs Native Functions
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def product_tier(price):
    return "Premium" if price > 50 else "Regular"

# Using UDF
products_dataframe.withColumn("tier_udf", product_tier("price")).show()

# Native
products_dataframe.withColumn(
    "tier_native",
    F.when(F.col("price") > 50, "Premium").otherwise("Regular")
).show()